# Check question-generation quality on OUR dataset topics

For topics from our dataset, the model writes **new** easy / medium / hard MCQs from the topic's passage,
and each one is printed next to the **closest existing dataset question** for that topic, so you can see whether the model
writes something new or just copies.

- **Test-split topics** (6, two per branch): the model never trained on these topics. This is the real quality check.
- **Train-split topics** (3, one per branch): it trained on these, so they show whether it just memorised.

Each MCQ shows: **grounding** (how much of the question + answer comes from the passage) and **similarity** to the closest
dataset question (0 = totally new, 1 = exact copy; above 0.7 is marked COPY).
At the end there is a summary table, and a cell where you can pick any topic from the dataset yourself.

**Setup:** GPU T4, Internet ON. Add Input: `btech-mcq-dataset` **and** your fine-tuning notebook's output.
Run the first 5 cells once (loads the model, about 2 minutes), then use the **YOUR OWN INPUT** cell as often as you like.

In [ ]:
!pip install -q -U "transformers>=4.46,<5"

In [ ]:
# Works on Kaggle AND Google Colab.
# Colab: put flan_t5_btech_mcq_model.zip and btech_mcq_dataset_clean.zip in Google Drive folder "adaptive-quiz".
import os, glob, zipfile
if os.path.exists("/kaggle/input"):
    ROOT = "/kaggle/input"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/adaptive-quiz"
    zips = glob.glob(f"{DRIVE_DIR}/*.zip")
    assert zips, f"no zip files in {DRIVE_DIR}: upload the model zip and the dataset zip there"
    ROOT = "/content/work"
    for z in zips:
        out = os.path.join(ROOT, os.path.basename(z)[:-4])
        if not os.path.exists(out):
            print("unzipping", z); zipfile.ZipFile(z).extractall(out)
print("data root:", ROOT)

In [ ]:
import glob, json, os, re, random, collections, torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
dev = "cuda" if torch.cuda.is_available() else "cpu"
cfgs = sorted(glob.glob(f"{ROOT}/**/train_config.json", recursive=True), key=lambda p: "/final/" not in p)
assert cfgs, "trained model not found: add the training notebook's output as an input"
model_dir = os.path.dirname(cfgs[0]); CFG = json.load(open(cfgs[0]))
tok = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(dev).eval()
model.config.use_cache = True

def find(split):
    hits = sorted(glob.glob(f"{ROOT}/**/{split}.jsonl", recursive=True), key=lambda p: ("clean" not in p, p))
    assert hits, f"{split}.jsonl not found: add the btech-mcq-dataset input"
    return hits[0]
rows = []
for split in ["train", "val", "test"]:
    for line in open(find(split), encoding="utf-8"):
        r = json.loads(line); r["split"] = split; rows.append(r)
data = pd.DataFrame(rows)
topics = data.groupby(["split", "branch", "subject", "topic"]).agg(context=("context", "first"),
                                                                   questions=("question", list)).reset_index()
print("loaded", model_dir, "|", len(data), "dataset questions,", len(topics), "topics")

In [ ]:
import re

LETTERS = "ABCD"


def build_input(branch, subject, topic, difficulty, context):
    return (f"generate mcq | branch: {branch} | subject: {subject} | "
            f"topic: {topic} | difficulty: {difficulty} | context: {context}")


_PATTERN = re.compile(
    r"question:\s*(?P<q>.+?)\s*options:\s*"
    r"\(A\)\s*(?P<A>.+?)\s*\(B\)\s*(?P<B>.+?)\s*\(C\)\s*(?P<C>.+?)\s*\(D\)\s*(?P<D>.+?)\s*"
    r"answer:\s*(?P<ans>[ABCD])\b\s*(?:explanation:\s*(?P<exp>.*))?$",
    re.S)


def parse_mcq(text):
    """Parse model output into a dict, or return None if the format is broken."""
    m = _PATTERN.search(text.strip())
    if not m:
        return None
    options = [m.group(l).strip() for l in LETTERS]
    norm = [re.sub(r"\s+", " ", o.lower()) for o in options]
    if any(not o for o in options) or len(set(norm)) < 4:
        return None
    return {"question": m.group("q").strip(), "options": options,
            "answer": m.group("ans"), "explanation": (m.group("exp") or "").strip()}

In [ ]:
STOP = set("the a an of to in is are and or for on with by as at be this that which from it its was were what how why when".split())
def content(s): return {w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in STOP and len(w) > 2}
def jacc(a, b):
    a, b = content(a), content(b)
    return len(a & b) / max(1, len(a | b))

@torch.no_grad()
def generate_mcq(branch, subject, topic, difficulty, context, n_cand=8, temperature=0.7):
    """Sample n_cand candidates, keep those with 4 distinct options, return the best grounded one (or None)."""
    prompt = build_input(branch, subject, topic, difficulty, context)
    enc = tok([prompt], return_tensors="pt", truncation=True, max_length=CFG["max_source_len"]).to(dev)
    ids = model.generate(**enc, do_sample=True, top_p=0.92, temperature=temperature, num_return_sequences=n_cand,
                         max_new_tokens=CFG["max_target_len"], no_repeat_ngram_size=4)
    best, ctx = None, content(context)
    for t in tok.batch_decode(ids, skip_special_tokens=True):
        p = parse_mcq(t)
        if not p:
            continue
        w = content(p["question"] + " " + p["options"]["ABCD".index(p["answer"])])
        p["grounding"] = round(len(w & ctx) / max(1, len(w)), 2)
        if best is None or p["grounding"] > best["grounding"]:
            best = p
    if best:   # the model prefers putting the answer at C/D, so shuffle the options to remove position bias
        correct = best["options"]["ABCD".index(best["answer"])]
        random.shuffle(best["options"])
        best["answer"] = "ABCD"[best["options"].index(correct)]
    return best

summary = []
def show(row):
    print("=" * 100, f"\n[{row.split.upper()} topic] {row.branch} | {row.subject} | {row.topic}")
    for d in ["easy", "medium", "hard"]:
        p = generate_mcq(row.branch, row.subject, row.topic, d, row.context)
        if p is None:
            print(f"\n[{d}] NO VALID MCQ"); summary.append({"split": row.split, "difficulty": d, "valid": 0}); continue
        sims = [(jacc(p["question"], q), q) for q in row.questions]
        sim, closest = max(sims)
        flag = "  <-- COPY" if sim > 0.7 else ""
        print(f"\n[{d}] grounding {p['grounding']} | similarity to dataset {sim:.2f}{flag}")
        print("MODEL:  ", p["question"]); [print(f"   ({l}) {o}") for l, o in zip("ABCD", p["options"])]
        print("   answer:", p["answer"], "|", p["explanation"])
        print("DATASET closest:", closest)
        summary.append({"split": row.split, "difficulty": d, "valid": 1, "grounding": p["grounding"],
                        "similarity": round(sim, 2), "copy": int(sim > 0.7)})

## ✍️ YOUR OWN INPUT: type branch, subject, topic and context, then run this cell

Edit the 4 values below and press **Shift+Enter**. Re-run as many times as you like (use the interactive session, not Save Version).
It shows the exact model input (same format as training), then **2 different MCQs per difficulty**, each with options,
correct answer and explanation. If the topic exists in our dataset, it also shows how similar each question is to our dataset questions.

In [ ]:
BRANCH  = "CSE"
SUBJECT = "Operating Systems"
TOPIC   = "Deadlock"
CONTEXT = """A deadlock is a situation where a set of processes are blocked because each process is holding a resource
and waiting for another resource held by some other process. Four conditions must hold at the same time for a deadlock:
mutual exclusion, hold and wait, no preemption and circular wait. Deadlocks can be handled by prevention, which breaks one
of the four conditions, by avoidance, such as the Banker's algorithm, which grants a request only if the system stays in a
safe state, or by detection and recovery, where the system finds cycles in the wait-for graph and then aborts a process or
preempts resources. Many operating systems simply ignore deadlocks, which is called the ostrich approach."""
DIFFICULTIES = ["easy", "medium", "hard"]
N_PER_DIFFICULTY = 2

print("MODEL INPUT:", build_input(BRANCH, SUBJECT, TOPIC, "<difficulty>", " ".join(CONTEXT.split()))[:300], "...\n")
known = data[data.topic.str.lower() == TOPIC.lower()].question.tolist()
for d in DIFFICULTIES:
    seen = set()
    for k in range(N_PER_DIFFICULTY * 3):           # a few extra tries to get distinct questions
        p = generate_mcq(BRANCH, SUBJECT, TOPIC, d, " ".join(CONTEXT.split()))
        if p is None or p["question"].lower() in seen:
            continue
        seen.add(p["question"].lower())
        sim = max([jacc(p["question"], q) for q in known], default=None)
        extra = f" | similarity to dataset {sim:.2f}" if sim is not None else " | topic not in dataset"
        print(f"[{d} #{len(seen)}] grounding {p['grounding']}{extra}")
        print(p["question"]); [print(f"   ({l}) {o}") for l, o in zip("ABCD", p["options"])]
        print("   correct answer:", p["answer"], "->", p["options"]["ABCD".index(p["answer"])])
        print("   explanation:", p["explanation"], "\n")
        if len(seen) == N_PER_DIFFICULTY:
            break
    if not seen:
        print(f"[{d}] no valid MCQ, try a longer or clearer context\n")

In [ ]:
random.seed(7)
picked = []
for split, per_branch in [("test", 2), ("train", 1)]:
    for b in ["CSE", "DSAI", "ECE"]:
        pool = topics[(topics.split == split) & (topics.branch == b)]
        picked += [pool.iloc[i] for i in random.sample(range(len(pool)), per_branch)]
for row in picked:
    show(row)

s = pd.DataFrame(summary)
print("\nSUMMARY (valid = share of requests that gave a valid MCQ)")
print(s.groupby("split").agg(requests=("valid", "size"), valid=("valid", "mean"), grounding=("grounding", "mean"),
                             similarity=("similarity", "mean"), copies=("copy", "sum")).round(2).to_string())

## Pick any topic from the dataset yourself

In [ ]:
# type part of a topic name, e.g. "deadlock", "fourier", "dropout"
SEARCH = "deadlock"
hits = topics[topics.topic.str.contains(SEARCH, case=False)]
print(hits[["split", "branch", "subject", "topic"]].to_string())
for row in hits.head(2).itertuples():
    show(row)